In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import  ChatOllama
from typing import TypedDict, Literal
from pydantic import BaseModel,Field

In [2]:
model = ChatOllama(model="llama3.2")

In [3]:
class SentimentSchema(BaseModel): 
    sentiment: Literal['positive', 'negative'] = Field(description="sentiment of the review.")

In [22]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal[
        "UX",
        "Performance",
        "Bug",
        "Support",
        "Other"
    ] = Field(
        description="The category of issue mentioned in the review."
    )

    tone: Literal[
        "angry",
        "frustrated",
        "disappointed",
        "calm"
    ] = Field(
        description="The emotional tone expressed by the customer in the review."
    )

    urgency: Literal[
        "low",
        "medium",
        "high"
    ] = Field(
        description="How urgent the issue is based on the customer review."
    )

In [23]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model_schema = model.with_structured_output(DiagnosisSchema)

In [8]:
prompt = "What is the sentiment of the following review - The software is bad"
structured_model.invoke(prompt)

SentimentSchema(sentiment='negative')

In [41]:
class ReviewState(TypedDict): 
    review: str 
    sentiment: Literal['positive', 'negative']
    diagnosis: dict 
    response: str

In [42]:
def find_sentiment(state: ReviewState):
    prompt =f"For the following review find out the sentiment \n {state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

In [43]:
def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:

    if state['sentiment'] == 'positive': 
        return 'positive_response'
    else: 
        return 'run_diagnosis'

In [44]:
def positive_response(state: ReviewState) -> ReviewState:
    prompt = f"""
    Write a warm thank-you message in response to this positive review.
    Also, kindly ask the user to leave feedback on our website.

    Review: "{state['review']}"
    """

    response = model.invoke(prompt).content

    state["response"] = response

    return state

In [45]:
def run_diagnosis(state: ReviewState):
    prompt = f"""
    Diagnose this negative customer review:
    {state['review']}

    Identify the issue type, tone, and urgency.
    """

    response = structured_model_schema.invoke(prompt)

    return {"diagnosis": response}

In [46]:
def negative_response(state: ReviewState):
    diagnosis = state["diagnosis"]

    prompt = f"""
    You are a customer support assistant.

    The user reported a '{diagnosis.issue_type}' issue.
    Their tone is '{diagnosis.tone}'.
    The urgency is '{diagnosis.urgency}'.

    Review: "{state['review']}"

    Write a polite, empathetic response. Acknowledge the problem,
    apologize for the inconvenience, and offer assistance.
    """

    response = model.invoke(prompt).content

    return {"response": response}

In [47]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)



#edges 
graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)
graph.add_edge("positive_response", END)
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("negative_response", END)

workflow = graph.compile()

In [ ]:
inital_state = {
    "review": "The produce was really bad"
}
workflow.invoke(inital_state)